# P3 · FAQ y repaso

**EDA, calidad y Parquet con DuckDB**

Este cuaderno no sustituye al enunciado de la tarea. Reúne las dudas que más
aparecen en el laboratorio, las comprobaciones que conviene saber explicar y un
repaso corto antes de entregar.

**Entorno:** `sbd-lab`. Ejecuta las celdas en orden.


## Cómo usar este cuaderno

- Ejecuta las celdas de arriba abajo.
- Las celdas leen `data/sales_raw.csv`; no sobrescriben el CSV.
- La celda de Parquet escribe en una carpeta temporal, no en `data/curated/`.
- Las preguntas del final son de repaso; no son una entrega nueva.


## Mapa del laboratorio

| Columna | Qué significa | Tipo esperado |
|---|---|---|
| `fecha` | Día de la venta | Fecha |
| `tienda_id` | Identificador de tienda | Texto |
| `sku` | Código de producto | Texto |
| `canal` | Origen de la venta: `tienda`, `web` o `app` | Categoría |
| `unidades` | Unidades vendidas | Entero `>= 0` |
| `precio` | Precio unitario en euros | Número `> 0` |
| `importe` | Importe total de la línea | Número `≈ unidades × precio` |

**Una fila es una línea de venta**, no un pedido.


## Materiales del laboratorio

Todo lo necesario está en la carpeta del laboratorio. Si te falta algo, usa el
ZIP publicado en Classroom y descomprímelo en una carpeta de trabajo.

| Archivo | Para qué sirve |
|---|---|
| `README.md` | Arranque, comandos y entregables. |
| `data/sales_raw.csv` | Dataset con incidencias intencionadas. |
| `gen_sales.py` | Regenera el CSV; no lo ejecutes si ya tienes datos. |
| `eda_calidad_medidas.ipynb` | Cuaderno de perfilado y métricas. |
| `limpieza_y_parquet.ipynb` | Cuaderno de limpieza y exportación. |
| `eda_quality.py` | Solución de referencia; no la uses como atajo. |
| `UD1_Informe_Calidad_template.md` | Plantilla del informe. |
| `quality_metrics_template.csv` | Matriz de métricas antes/después. |
| `duckdb_checks_template.sql` | Comprobaciones SQL de calidad. |
| `quality_report_template.json` | Estructura del informe automático. |
| `requirements.txt` | Dependencias del laboratorio. |

La carpeta `data/curated/` se genera al exportar; no es un archivo de partida.


In [1]:
from pathlib import Path

requeridos = [
    "README.md",
    "data/sales_raw.csv",
    "eda_calidad_medidas.ipynb",
    "limpieza_y_parquet.ipynb",
    "UD1_Informe_Calidad_template.md",
    "quality_metrics_template.csv",
    "duckdb_checks_template.sql",
]
for r in requeridos:
    print("OK   " if Path(r).exists() else "FALTA", r)


OK    README.md
OK    data/sales_raw.csv
OK    eda_calidad_medidas.ipynb
OK    limpieza_y_parquet.ipynb
OK    UD1_Informe_Calidad_template.md
OK    quality_metrics_template.csv
OK    duckdb_checks_template.sql


## FAQ 1 · ¿Por qué pandas no lee bien `precio`?

Porque hay una mezcla de decimales con punto y con coma (`11.86` y `14,00`).
Si pandas infiere el tipo, la columna puede quedar como texto y los cálculos
fallan.

La lectura robusta es: leer todo como texto, limpiar espacios y convertir después.


In [2]:
from pathlib import Path
import pandas as pd

RUTA = Path("data/sales_raw.csv")
df = pd.read_csv(RUTA, dtype=str, keep_default_na=False)
df = df.apply(lambda col: col.str.strip())
print("Filas y columnas:", df.shape)
df.head()


Filas y columnas: (86536, 7)


,fecha,tienda_id,sku,canal,unidades,precio,importe
0,2024-06-01,S005,SKU00203,web,4,11.86,47.44
1,2024-06-01,S014,SKU00215,tienda,2,72.32,144.64
2,2024-06-01,S002,SKU00296,web,2,4.23,8.46
3,2024-06-01,S019,SKU00158,web,1,31.13,31.13
4,2024-06-01,S019,SKU00031,web,2,13.57,27.14


## FAQ 2 · ¿Cómo perfilo el dataset?

Antes de limpiar, describe qué hay: nulos, duplicados exactos, valores de
`canal` y rangos básicos. No empieces borrando.


In [3]:
print("Nulos por columna:")
print(df.replace("", pd.NA).isna().sum())

print("\nDuplicados exactos:", df.duplicated().sum())

print("\nCanales:")
print(df["canal"].value_counts().head(10))


Nulos por columna:
fecha           0
tienda_id       0
sku             0
canal           0
unidades        0
precio       1720
importe      2421
dtype: int64

Duplicados exactos: 856

Canales:
canal
tienda    37760
web       33553
app       12627
Web         537
WEB         534
W           522
online      511
wEb         492
Name: count, dtype: int64


## FAQ 3 · ¿Cómo convierto los tipos?

- `precio` e `importe`: cambiar coma por punto y usar `pd.to_numeric`.
- `unidades`: entero, permitiendo nulos.
- `fecha`: `pd.to_datetime` con `errors="coerce"` para marcar las inválidas.

Un texto que parece número no es un número.


In [4]:
df["precio"] = pd.to_numeric(
    df["precio"].str.replace(",", ".", regex=False), errors="coerce"
)
df["importe"] = pd.to_numeric(
    df["importe"].str.replace(",", ".", regex=False), errors="coerce"
)
df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

df.dtypes


fecha        datetime64[us]
tienda_id               str
sku                     str
canal                   str
unidades              int64
precio              float64
importe             float64
dtype: object

## FAQ 4 · ¿Qué métricas mínimas debo calcular?

Siempre con numerador y denominador explícitos:

- **Completitud:** valores no nulos / total de filas.
- **Unicidad:** filas únicas / total, o duplicados según una clave.
- **Validez:** valores dentro del dominio o rango / total.
- **Consistencia:** `importe ≈ unidades × precio` dentro de una tolerancia.

Una métrica sin granularidad ni denominador no significa nada.


In [5]:
total = len(df)

completitud_precio = df["precio"].notna().sum() / total

base = df.dropna(subset=["unidades", "precio", "importe"])
consistencia = (
    (
        (base["importe"] - base["unidades"] * base["precio"]).abs()
        <= 0.02 * (base["unidades"] * base["precio"]).abs()
    ).sum()
    / len(base)
)

print(f"Completitud de precio: {completitud_precio:.3f}")
print(f"Consistencia de importe (denominador {len(base)}): {consistencia:.3f}")


Completitud de precio: 0.980
Consistencia de importe (denominador 84115): 0.992


## FAQ 5 · ¿Qué hago con las variantes de `canal`?

Hay valores como `Web`, `WEB`, `W`, `online` o ` wEb `. No son canales nuevos:
son el mismo canal mal escrito. Normaliza antes de medir validez.

Cuidado: normalizar no es inventar. Si aparece un valor que no puedes mapear,
déjalo marcado y documentado.


In [6]:
mapa_canal = {"w": "web", "online": "web"}

df["canal_norm"] = (
    df["canal"].str.lower().str.strip().replace(mapa_canal)
)

df["canal_norm"].value_counts()


canal_norm
tienda    37760
web       36149
app       12627
Name: count, dtype: int64

## FAQ 6 · ¿Qué hago con un precio nulo?

Depende de la regla y del contexto. Opciones habituales:

1. **Imputar** con la mediana del mismo SKU, solo si ese SKU tiene otros precios.
2. **Dejar nulo** y marcar la fila si no hay información suficiente.
3. **Separar para revisión** si el precio es imprescindible.

Lo que no deberías hacer es inventar un precio sin dejar rastro. Si imputas,
documenta la regla y conserva una marca.


## FAQ 7 · ¿Qué hago con unidades negativas o cero?

- Una unidad negativa suele ser una devolución, un ajuste o un error de captura.
- Un cero puede ser una fila sin venta real.

No las borres por reflejo. Márcalas, decide si se excluyen del análisis y
explica el impacto. Si las eliminas, indica cuántas filas pierdes.


In [7]:
negativas = df["unidades"] < 0
cero = df["unidades"] == 0

print("Unidades negativas:", negativas.sum())
print("Unidades cero:", cero.sum())
df.loc[negativas, ["unidades", "precio", "importe"]].head()


Unidades negativas: 719
Unidades cero: 13210


,unidades,precio,importe
50,-1,18.14,NaN
212,-4,27.77,NaN
221,-3,22.61,NaN
356,-2,26.34,NaN
710,-3,6.37,NaN


## FAQ 8 · ¿Qué hago con fechas de 2099?

Son fechas futuras improbables. No las trates como ventas reales: márcalas como
inválidas o fuera de rango y exclúyelas del dataset curado con trazabilidad.


In [8]:
hoy = pd.Timestamp.today().normalize()
futuras = df["fecha"] > hoy

print("Fechas futuras:", futuras.sum())
print(df.loc[futuras, "fecha"].value_counts().head())


Fechas futuras: 161
fecha
2099-01-01    161
Name: count, dtype: int64


## FAQ 9 · ¿Todo identificador repetido es un duplicado?

No. Distingue:

- **Duplicado exacto:** todas las columnas coinciden.
- **Duplicado de clave:** coinciden las columnas que definen la granularidad,
  por ejemplo `fecha`, `tienda_id`, `sku` y `canal`.

Antes de deduplicar, define la clave. Si no, puedes borrar ventas legítimas.


In [9]:
clave = ["fecha", "tienda_id", "sku", "canal_norm"]

print("Duplicados exactos:", df.duplicated().sum())
print("Duplicados de clave:", df.duplicated(subset=clave).sum())


Duplicados exactos: 856
Duplicados de clave: 2723


## FAQ 10 · ¿Borro los valores extremos?

No automáticamente. Un outlier es un valor extremo, no necesariamente un error.
Primero investígalo: mira `unidades`, `precio` e `importe` de esa fila y comprueba
si la operación tiene sentido.

Puedes usar el criterio de Tukey: `Q3 + 1,5 × IQR`. Sirve para **señalar**, no
para borrar sin explicación.


In [10]:
q1, q3 = df["importe"].quantile([0.25, 0.75])
iqr = q3 - q1
limite = q3 + 1.5 * iqr

print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}  límite={limite:.2f}")
df.loc[df["importe"] > limite, ["unidades", "precio", "importe"]].head()


Q1=10.62  Q3=44.19  IQR=33.57  límite=94.55


,unidades,precio,importe
1,2,72.32,144.64
16,2,49.67,99.34
34,4,26.11,104.44
153,2,48.31,96.62
162,3,34.06,102.18


## FAQ 11 · ¿Cómo exporto a Parquet particionado?

Crea columnas de año y mes y usa `partition_cols`. Cada partición se guarda en
una carpeta `anio=YYYY/mes=MM/`.

No particiones por una columna que no vas a filtrar. Y no generes cientos de
carpetas diminutas: también tiene coste.


In [11]:
import tempfile

ruta_demo = Path(tempfile.mkdtemp(prefix="repaso_p3_parquet_"))
limpio = df[df["fecha"].notna() & (df["fecha"] <= hoy)].copy()
limpio["anio"] = limpio["fecha"].dt.year
limpio["mes"] = limpio["fecha"].dt.month

limpio.to_parquet(
    ruta_demo,
    partition_cols=["anio", "mes"],
    index=False,
)

print(ruta_demo)


/tmp/repaso_p3_parquet_r8gs9ehl


## FAQ 12 · ¿Cómo consulto el Parquet con DuckDB?

DuckDB puede leer un glob de particiones sin cargar todo en memoria. Usa la ruta
completa o relativa al directorio de trabajo.


In [12]:
import duckdb

patron = f"{ruta_demo.as_posix()}/anio=*/mes=*/*.parquet"
duckdb.sql(
    f"SELECT COUNT(*) AS filas FROM read_parquet('{patron}')"
).show()


┌───────┐
│ filas │
│ int64 │
├───────┤
│ 86375 │
└───────┘



## FAQ 13 · ¿La comparación CSV/Parquet es un benchmark?

No. Es una **medición de aula**: un archivo, un equipo y una consulta. Puedes
decir «en este equipo, para esta consulta, Parquet ocupó menos y leyó solo las
columnas necesarias». No puedes decir «Parquet es siempre más rápido» ni
generalizar a otros volúmenes.

Si mides, repite una vez para separar caché fría de caliente y anota la consulta.


## FAQ 14 · Errores frecuentes

| Síntoma | Causa probable | Qué hacer |
|---|---|---|
| `No module named pyarrow` | Falta el motor de Parquet | Usa el entorno `sbd-lab` del laboratorio |
| `FileNotFoundError` | Estás fuera de la carpeta del laboratorio | `cd` a `lab_p3_eda_calidad_duckdb/` |
| `precio` es `object` | Decimales con coma | `str.replace(",", ".")` y `pd.to_numeric` |
| `KeyError: 'anio'` | No creaste las columnas de partición | Créalas antes de `to_parquet` |
| `read_parquet` no encuentra archivos | Glob incorrecto | Comprueba `anio=*/mes=*/*.parquet` |
| Se sobrescribe `data/curated/` | Exportación sobre el dataset del laboratorio | Escribe en una copia o carpeta temporal |


## Decisiones que debes justificar

| Incidencia | Decisión posible | Qué documentar |
|---|---|---|
| Precio nulo | Imputar, dejar nulo o separar | Regla y nº de filas afectadas |
| Unidades negativas | Marcar, excluir o revisar | Motivo y filas perdidas |
| Fecha 2099 | Marcar como futura y excluir | Cuántas filas y por qué |
| Duplicado exacto | Eliminar | Cuántas filas y qué criterio |
| `canal` mal escrito | Normalizar | Mapa aplicado |
| `importe` incoherente | Marcar, corregir o conservar | Tolerancia y trazabilidad |


## Mini-repaso

Responde sin mirar las celdas anteriores:

1. ¿Qué representa una fila de `sales_raw.csv`?
2. ¿Cuál es el denominador de la completitud de `precio`?
3. ¿Por qué `canal` puede parecer inválido antes de normalizarlo?
4. ¿Cuándo eliminarías un duplicado y cuándo no?
5. ¿Qué demuestra y qué no demuestra la comparación CSV/Parquet?


In [13]:
# Escribe aquí tus respuestas o tus comprobaciones.


## Checklist antes de entregar

- [ ] Perfilado inicial y métricas **antes**.
- [ ] Al menos una métrica de completitud, una de unicidad, una de validez y una
      de consistencia, con denominadores.
- [ ] Decisiones de limpieza justificadas y trazables.
- [ ] Métricas **después** y diferencia en puntos porcentuales.
- [ ] Parquet particionado y consultas DuckDB reproducibles.
- [ ] Comparación CSV/Parquet medida y explicada sin generalizar.
- [ ] Mini-informe con decisiones calidad ↔ coste.


## Rúbrica (/10)

| Criterio | Puntos |
|---|---:|
| EDA completo: perfilado, distribuciones, outliers y lectura de gráficos | 2 |
| Métricas de calidad antes/después, bien interpretadas | 2 |
| Limpieza, tipado, imputación/etiquetado y decisiones justificadas | 2 |
| Parquet, particionado, compresión y comparación CSV vs Parquet | 2 |
| Consultas DuckDB reproducibles e informe claro sobre impacto calidad-coste | 2 |
